# UCS420 Lab Assignment 9 – NLP Text Preprocessing

Name: Abhinandan Goyal  
Roll No.: 1024170176

Run the cells from top to bottom.

In [ ]:
ROLL_NO = 1024170176
NAME = "Abhinandan Goyal"

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:

all_results = []
rows = []

for _, ticket_row in df.iterrows():
    result = process_ticket(ticket_row["Ticket_Text"])
    all_results.append(result)
    rows.append({
        "Ticket_ID": ticket_row["Ticket_ID"],
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique tokens": len(set(result["tokens"])),
        "Final tokens": len(result["final"]),
    })

q1_df = pd.DataFrame(rows)
count_cols = ["Sentences", "Tokens", "Punctuation tokens", "Stopwords", "Unique tokens", "Final tokens"]
total_row = {"Ticket_ID": "TOTAL"}
for col in count_cols:
    total_row[col] = int(q1_df[col].sum())
total_row["Unique tokens"] = len(set(token for result in all_results for token in result["tokens"]))
q1_df = pd.concat([q1_df, pd.DataFrame([total_row])], ignore_index=True)
display(q1_df)

focus_id = FOCUS[0] if FOCUS and FOCUS[0] in set(df["Ticket_ID"]) else df.iloc[0]["Ticket_ID"]
focus_index = df.index[df["Ticket_ID"] == focus_id][0]
focus_result = all_results[focus_index]
print(f"\nDetailed preprocessing for {focus_id}: {df.loc[focus_index, 'Ticket_Text']}")
print("Tokens:", focus_result["tokens"])
print("Punctuation tokens:", focus_result["punct"])
print("Stopwords:", focus_result["stops"])
print("Final tokens:", focus_result["final"])

**Observation:** Some contractions get split into parts like `ca` and `n't`. These parts can still appear in the final tokens because they may not be treated as stopwords or punctuation.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
import re
from collections import Counter

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]
patterns = {
    "Words > 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers (including decimals/versions)": r"(?<![\w.])\d+(?:,\d{3})*(?:\.\d+)*(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words starting with a vowel": r"\b[AEIOUaeiou][a-zA-Z]*\b",
}

print("EXTRACTION FROM RAW TICKETS")
for label, pattern in patterns.items():
    print(f"\n--- {label} ---")
    for _, ticket_row in df.iterrows():
        print(f"{ticket_row['Ticket_ID']}: {re.findall(pattern, ticket_row['Ticket_Text'])}")

email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
url_pattern = r"https?://[^\s<>]+|www\.[^\s<>]+"
phone_pattern = r"(?<!\w)(?:\+\d{1,3}[- ]?)?(?:\d{5}[- ]\d{5}|\d{4}[- ]\d{3}[- ]\d{4}|\d{3}[- ]\d{3}[- ]\d{4})(?!\w)"

def replace_patterns(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)
    return text

print("\nREPLACEMENT TESTS ON TEST_LINES")
for line in TEST_LINES:
    print(replace_patterns(line))
print("\nREPLACEMENT ON RAW TICKETS")
for _, ticket_row in df.iterrows():
    print(f"{ticket_row['Ticket_ID']}: {replace_patterns(ticket_row['Ticket_Text'])}")

# Estimate capitalized words occurring at sentence starts, then report the observed count.
capitalized = []
sentence_initial_capitalized = []
for text in df["Ticket_Text"]:
    capitalized.extend(re.findall(patterns["Capitalized words"], text))
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        match = re.match(r"\s*([A-Z][a-zA-Z]*)\b", sentence)
        if match:
            sentence_initial_capitalized.append(match.group(1))
# A practical heuristic: count sentence-initial capitalized occurrences whose lowercase form also
# appears elsewhere in lowercase, suggesting capitalization may be due to sentence position.
lowercase_noninitial_words = set()
for text in df["Ticket_Text"]:
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        first = re.match(r"\s*[A-Z][a-zA-Z]*\b", sentence)
        remainder = sentence[first.end():] if first else sentence
        lowercase_noninitial_words.update(re.findall(r"\b[a-z]+\b", remainder))
likely_sentence_capitalization = [word for word in sentence_initial_capitalized if word.lower() in lowercase_noninitial_words]
print(f"\nCapitalized-word matches: {len(capitalized)}")
print(f"Sentence-initial capitalized occurrences: {len(sentence_initial_capitalized)}")
print(f"Likely capitalized only because of sentence position (heuristic): {len(likely_sentence_capitalization)}")
print("Likely examples:", likely_sentence_capitalization[:15])

**Observation:** The regular expressions extract the required words and numbers, and replace emails, URLs and phone numbers with labels. Capitalized words are not always names; some are just at the beginning of a sentence, so this method can give a rough estimate.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
from nltk.tokenize import RegexpTokenizer, word_tokenize

token_pattern = (
    r"https?://[^\s<>]+|www\.[^\s<>]+|"
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|"
    r"(?<!\w)(?:\+\d{1,3}[- ]?)?(?:\d{5}[- ]\d{5}|\d{4}[- ]\d{3}[- ]\d{4}|\d{3}[- ]\d{3}[- ]\d{4})(?!\w)|"
    r"\b\d+(?:,\d{3})*(?:\.\d+)+\b|"
    r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b|"
    r"\b[A-Za-z]+(?:['’][A-Za-z]+)+\b|"
    r"\b[A-Za-z]+\b|"
    r"\d+(?:,\d{3})*(?:\.\d+)?|"
    r"[^\w\s]"
)
my_tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    """Tokenize while preserving URLs, emails, phone numbers, versions, decimals, hyphens and contractions."""
    return my_tokenizer.tokenize(text)

print("TEST_LINES: custom tokenizer vs NLTK word_tokenize")
for i, line in enumerate(TEST_LINES, start=1):
    print(f"\nTest line {i}: {line}")
    print("my_tokenize:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))

comparison_rows = []
for _, ticket_row in df.iterrows():
    text = ticket_row["Ticket_Text"]
    custom_tokens = my_tokenize(text)
    nltk_tokens = word_tokenize(text)
    comparison_rows.append({
        "Ticket_ID": ticket_row["Ticket_ID"],
        "Custom tokens": custom_tokens,
        "word_tokenize tokens": nltk_tokens,
        "Custom count": len(custom_tokens),
        "NLTK count": len(nltk_tokens),
        "Different?": custom_tokens != nltk_tokens,
    })
q3_df = pd.DataFrame(comparison_rows)
display(q3_df)
different_ticket_count = int(q3_df["Different?"].sum())
print(f"Tickets with different token sequences: {different_ticket_count} / {len(q3_df)}")

**Observation:** The custom tokenizer keeps items like `isn't`, `state-of-the-art`, email addresses, URLs and version numbers together. The normal tokenizer may split some of these into multiple tokens, so the results can be different.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
import nltk
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

for pkg in ["averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

unique_final_tokens = list(dict.fromkeys(
    token.lower() for result in all_results for token in result["final"] if token.isalpha()
))
suffix_candidates = ["ing", "edly", "ed", "ly", "ies", "s"]
chosen_suffix = next((suffix for suffix in suffix_candidates
                      if sum(word.endswith(suffix) and len(word) > len(suffix) + 2
                             for word in unique_final_tokens) >= 2), "ing")
regex_stemmer = RegexpStemmer(chosen_suffix + r"$", min=3)
print("RegexpStemmer suffix selected:", repr(chosen_suffix))
print("Dataset words supporting this suffix:", [w for w in unique_final_tokens if w.endswith(chosen_suffix) and len(w) > len(chosen_suffix) + 2][:10])

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

pos_tags = nltk.pos_tag(unique_final_tokens)
rows = []
for word, tag in pos_tags:
    pos = get_wordnet_pos(tag)
    results = {
        "Porter Stem": porter.stem(word),
        "Lancaster Stem": lancaster.stem(word),
        "Regexp Stem": regex_stemmer.stem(word),
        "WordNet Lemma": lemmatizer.lemmatize(word, pos=pos),
    }
    if len(set(results.values())) > 1:
        rows.append({"Word": word, "POS Tag": tag, **results})
q4_df = pd.DataFrame(rows, columns=["Word", "POS Tag", "Porter Stem", "Lancaster Stem", "Regexp Stem", "WordNet Lemma"])
display(q4_df)
print("Unique alphabetic final tokens:", len(unique_final_tokens))
print("Words with differing outputs:", len(q4_df))


if not q4_df.empty:
    print("Examples from the differing-output table:")
    display(q4_df.head(10))

**Observation:** Porter and Lancaster sometimes give different stems for the same word. Lancaster is usually more aggressive, so it can shorten words too much. Lemmatization uses the word's meaning and part of speech, so its output can look more like a normal word.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
from nltk.util import bigrams
from math import log

S1 = [token.lower() for result in all_results for token in result["tokens"]]
S2 = [token.lower() for result in all_results for token in result["tokens"] if not is_punct(token)]
S3 = [token.lower() for result in all_results for token in result["final"]]
S4 = [porter.stem(token) for token in S3]

corpora = {"S1 (All tokens)": S1, "S2 (No punctuation)": S2,
           "S3 (Final tokens)": S3, "S4 (S3 after Porter stemming)": S4}
stats_rows = []
for corpus_name, tokens in corpora.items():
    stats_rows.append({"Corpus": corpus_name, "Total tokens": len(tokens),
                       "Unique tokens": len(set(tokens)),
                       "TTR": round(len(set(tokens)) / len(tokens), 4) if tokens else 0})
stats_df = pd.DataFrame(stats_rows)
display(stats_df)

print("\nTop 10 words in S1:")
display(pd.DataFrame(Counter(S1).most_common(10), columns=["Word", "Frequency"]))
print("Top 10 words in S3:")
display(pd.DataFrame(Counter(S3).most_common(10), columns=["Word", "Frequency"]))

freq_s3 = Counter(S3).most_common()
ranks = list(range(1, len(freq_s3) + 1))
frequencies = [frequency for _, frequency in freq_s3]
plt.figure(figsize=(8, 5))
plt.plot([log(rank) for rank in ranks], [log(freq) for freq in frequencies], marker="o", linestyle="-")
plt.xlabel("log(Rank)")
plt.ylabel("log(Frequency)")
plt.title("Log(Rank) vs Log(Frequency) for S3")
plt.grid(True)
plt.tight_layout()
plt.show()

print("Top 5 bigrams in S1:")
display(pd.DataFrame(Counter(bigrams(S1)).most_common(5), columns=["Bigram", "Frequency"]))
print("Top 5 bigrams in S3:")
display(pd.DataFrame(Counter(bigrams(S3)).most_common(5), columns=["Bigram", "Frequency"]))

**Observation:** TTR is calculated by dividing the number of unique tokens by the total number of tokens. Removing punctuation and stopwords changes the token count, and stemming combines some word forms. The exact change depends on the ticket data, so I used the values printed in the table.

---
## Q6. Interpretation
Answer the four questions using the results from the cells above.

In [ ]:

print("(a) TTR change from S1 to S4:")
display(stats_df)
row_s1 = stats_df.loc[stats_df["Corpus"] == "S1 (All tokens)"].iloc[0]
row_s4 = stats_df.loc[stats_df["Corpus"] == "S4 (S3 after Porter stemming)"].iloc[0]
print(f"In this dataset, TTR changes from {row_s1['TTR']} (S1) to {row_s4['TTR']} (S4). S1 includes punctuation and stopwords; S4 contains the final tokens after Porter stemming. Stemming merges some word forms, reducing vocabulary size relative to token count.")

print("\n(b) Example where tokenizers differ:")
example_line = next((line for line in TEST_LINES if my_tokenize(line) != word_tokenize(line)), TEST_LINES[0])
print("Text:", example_line)
print("my_tokenize:", my_tokenize(example_line))
print("word_tokenize:", word_tokenize(example_line))
print("Explanation: the custom regex keeps selected forms such as contractions, hyphenated words, versions, URLs, emails and phone numbers intact, while word_tokenize may split them.")

print("\n(c) Candidate over-stemming example from this dataset:")
over_candidates = q4_df[(q4_df["Lancaster Stem"] != q4_df["Word"]) &
                         (q4_df["Lancaster Stem"].str.len() <= q4_df["Word"].str.len() - 3)] if not q4_df.empty else q4_df
if not over_candidates.empty:
    candidate = over_candidates.iloc[0]
    print(f"Word: {candidate['Word']!r}; Lancaster output: {candidate['Lancaster Stem']!r}; Porter output: {candidate['Porter Stem']!r}; WordNet lemma: {candidate['WordNet Lemma']!r}")
    print("Use this as an over-stemming example only if the reduced form visibly loses useful meaning; briefly explain the loss.")
else:
    print("No clear over-stemming candidate was found automatically. Inspect the displayed Q4 table and choose an output that removes too much meaning.")

print("\n(d) Candidate where POS-aware lemmatization is more interpretable:")
lemma_candidates = q4_df[(q4_df["WordNet Lemma"] != q4_df["Word"]) &
                         (q4_df["WordNet Lemma"] != q4_df["Porter Stem"])] if not q4_df.empty else q4_df
if not lemma_candidates.empty:
    candidate = lemma_candidates.iloc[0]
    print(f"Word: {candidate['Word']!r}; POS: {candidate['POS Tag']}; Porter: {candidate['Porter Stem']!r}; WordNet lemma: {candidate['WordNet Lemma']!r}")
    print("Explain that WordNet uses the POS tag to return a dictionary form, whereas a stemmer mechanically strips or transforms suffixes.")
else:
    print("No clear example found automatically; inspect Q4 for a word whose POS-aware lemma is a meaningful dictionary form unlike its stem.")